The first step reads your granular Silver data and calculates the missing key meteorological factor: Dew Point Depression (the difference between actual air temperature and dew point temperature).

In [0]:
from pyspark.sql import functions as F

# 1. Load the Silver data (preserving the original string schema)
df_silver = spark.read.table("weather.silver.silver_weather")

# 2. Calculate Dew Point Depression by casting strings to double ONLY during the calculation.
# This keeps the original 'temperature_2m' and 'dew_point_2m' columns as strings in the DataFrame.
df_cloud_base = df_silver.withColumn(
    "dew_point_depression",
    F.round(F.col("temperature_2m").cast("double") - F.col("dew_point_2m").cast("double"), 2)
)


In [0]:
df_silver.describe()

The Multi-Condition Scoring Engine
Next, we evaluate every single hour against your specific criteria. We assign a binary score (1 for matching the condition, 0 for failing) across all five required constraints.

In [0]:
# Apply constraints by casting the string columns to double specifically inside the evaluation logic
# to prevent lexical string comparison errors (e.g., preventing "90" from being evaluated > "100")
df_cloud_scored = df_cloud_base \
    .withColumn("score_low_clouds", F.when(F.col("cloud_cover_low").cast("double").between(80, 100), 1).otherwise(0)) \
    .withColumn("score_sky_clear", F.when((F.col("cloud_cover_mid").cast("double") <= 20) & (F.col("cloud_cover_high").cast("double") <= 20), 1).otherwise(0)) \
    .withColumn("score_humidity", F.when(F.col("relative_humidity_2m").cast("double") >= 85, 1).otherwise(0)) \
    .withColumn("score_dew_point", F.when(F.col("dew_point_depression").between(0, 2), 1).otherwise(0)) \
    .withColumn("score_wind", F.when(F.col("wind_speed_10m").cast("double") < 8, 1).otherwise(0)) \
    .withColumn(
        "sea_of_clouds_index",
        F.col("score_low_clouds") + F.col("score_sky_clear") + F.col("score_humidity") + F.col("score_dew_point") + F.col("score_wind")
    )

Finally, we roll this data up to find the optimal viewing windows. We calculate how many hours per month met the perfect conditions (sea_of_clouds_index == 5) and save it to the catalog.

In [0]:
# Aggregate by casting the string columns before averaging
df_gold_cloud_view = df_cloud_scored.groupBy("year", "month", "weather").agg(
    F.sum(F.when(F.col("sea_of_clouds_index") == 5, 1).otherwise(0)).alias("perfect_viewing_hours"),
    F.round(F.avg(F.col("cloud_cover_low").cast("double")), 1).alias("avg_low_cloud_pct"),
    F.round(F.avg(F.col("relative_humidity_2m").cast("double")), 1).alias("avg_humidity_pct"),
    F.round(F.avg(F.col("wind_speed_10m").cast("double")), 1).alias("avg_wind_speed_kmh")
).orderBy(F.col("perfect_viewing_hours").desc())

# Save directly into the 3-level Gold catalog schema
(df_gold_cloud_view.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("weather.gold.gold_cloud_view_summary"))